# Mr.A-500M — Training on Colab (Tesla T4)

Trains the 505M-parameter Mr.A model from random initialization on the frozen
V19.2 corpus (164.9M train tokens, uint16 binaries on Google Drive).

**Plan:** BF16 + gradient checkpointing + micro-batch 1 + grad-accum 64
(262k tokens per optimizer step), 5 epochs ≈ 3,145 optimizer steps.

**Time estimate:** roughly 30–45 hours of T4 compute in total.
Colab sessions disconnect, so this notebook is built for **multiple sessions**:
checkpoints are saved to Drive and training resumes with `--resume`.


In [ ]:
import torch
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
!nvidia-smi -L

## 1. Mount Drive, install deps, fetch code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install sentencepiece safetensors tqdm
!rm -rf "/content/Mr.A-AI-model"
!git clone https://github.com/arkarsoe123/Mr.A-AI-model.git "/content/Mr.A-AI-model" 2>&1 | tail -1
import os
code_dir = "/content/Mr.A-AI-model/mr-a-500m"
if not os.path.exists(os.path.join(code_dir, "src", "train.py")):
    code_dir = "/content/drive/MyDrive/MrA_AI/code/mra-500m"
    print("GitHub \u0e21\u0e3e\u0e38 code \u0e21\u0e39\u0e3e\u0e59 \u2192 Drive fallback:", code_dir)
    print("mra-500m folder \u0e42\u0e2d Drive MrA_AI/code/ \u0e11\u0e32\u0e3a\u0e24\u0e3a\u0e37\u0e38\u0e24\u0e4a\u0e44\u0e92\u0e3a\u0e1e\u0e42\u0e1b\u0e3a")
os.chdir(code_dir)
!ls

## 2. Verify data + tokenizer paths

Expected Drive layout (from the V19.2 build):
- `/content/drive/MyDrive/MrA_AI/corpus/v19_2/bin/mra_v19_2_train.uint16.bin`
- `/content/drive/MyDrive/MrA_AI/corpus/v19_2/bin/mra_v19_2_val.uint16.bin`
- `/content/drive/MyDrive/MrA_AI/tokenizer/mra_v10_tokenizer/mra_tokenizer.model`

If any path is `False`, edit `config_500m.json` (the `data` section) to match
your Drive, then re-run the cell below.

In [ ]:
import os
paths = {
    "train_bin": "/content/drive/MyDrive/MrA_AI/corpus/v19_2/bin/mra_v19_2_train.uint16.bin",
    "val_bin":   "/content/drive/MyDrive/MrA_AI/corpus/v19_2/bin/mra_v19_2_val.uint16.bin",
    "tokenizer": "/content/drive/MyDrive/MrA_AI/tokenizer/mra_v10_tokenizer/mra_tokenizer.model",
}
for k, p in paths.items():
    print(os.path.exists(p), k, "->", p)

## 3. Train

First session: starts from scratch. Later sessions: **resume** from the latest
checkpoint (see next cell). Checkpoints + logs go to `out/mra_500m/` —
back them up to Drive after each session (cell 5).

In [ ]:
!python src/train.py --config config_500m.json --out-dir /content/drive/MyDrive/MrA_AI/runs/mra_500m

## 4. Resume (sessions 2, 3, ...)

Pick the newest checkpoint under your Drive runs folder and resume:
`--resume /content/drive/MyDrive/MrA_AI/runs/mra_500m/ckpt_epoch0.pt`
(or the latest `ckpt_stepNNNN.pt`).

In [ ]:
# Example — edit the checkpoint name to the newest one, then run:
# !python src/train.py --config config_500m.json \
#     --out-dir /content/drive/MyDrive/MrA_AI/runs/mra_500m \
#     --resume /content/drive/MyDrive/MrA_AI/runs/mra_500m/ckpt_epoch0.pt
print("edit + uncomment the cell above, then run it")

## 5. Watch the loss

In [ ]:
!tail -5 /content/drive/MyDrive/MrA_AI/runs/mra_500m/train_log.jsonl

## 6. Export (after training finishes)

Converts the final checkpoint to HuggingFace LLaMA format, ready for
GGUF conversion (`convert_hf_to_gguf.py`) and Q4_K_M quantization
(~300 MB) for Android on-device inference.

In [ ]:
# !python src/export_hf.py \
#     --ckpt /content/drive/MyDrive/MrA_AI/runs/mra_500m/ckpt_epoch4.pt \
#     --tokenizer /content/drive/MyDrive/MrA_AI/tokenizer/mra_v10_tokenizer/mra_tokenizer.model \
#     --out /content/drive/MyDrive/MrA_AI/exports/hf_mra_500m
print("uncomment when training is done")